# NCKH — V4: 111 Verified Hard Negatives + Qwen3-VL Detection Verification

## Mục tiêu

V4 tập trung vào câu hỏi gốc:

> **CountGD++ có đang đếm đúng loại đối tượng mà người dùng yêu cầu hay không?**

V4 mở rộng V3 bằng hai thay đổi:

1. Dùng benchmark mới **111 hard-negative pairs đã xác nhận target không tồn tại**:
   - 96 ảnh duy nhất;
   - 29 source classes;
   - 72 absent query targets;
   - 56 pair cũ (`V3_OLD_56`);
   - 55 pair mới (`V4_NEW_REVIEWED`).
2. Thay verifier bằng **Qwen3-VL-4B-Instruct**.

Pipeline:

```text
Image + Target
      ↓
CountGD++ candidate boxes
      ↓
Crop từng box (+ context)
      ↓
Qwen3-VL-4B detection verifier
      ↓
YES = giữ detection
NO  = loại detection
      ↓
Filtered count
```

## Hai nhánh đánh giá

- **Positive:** 96 ảnh duy nhất của benchmark, query đúng `source_class`; candidate boxes lấy từ cache V1.
- **Hard negative:** 111 image-query pairs, query là `query_target` đã được xác nhận absent; CountGD++ được chạy lại để lấy boxes.

## Quy tắc khoa học

- 111 hard-negative pairs chỉ dùng để **đánh giá**, không tune threshold/prompt trên chúng.
- Báo cáo riêng `OLD_56` và `NEW_55`; `NEW_55` là phần mở rộng mới quan trọng để kiểm tra tổng quát hóa.
- FSC-147 chỉ có point annotations, không có GT bounding box; positive box matching là **xấp xỉ**.
- Parser đã sửa lỗi V3 và chấp nhận cả `0|YES` và `ID0|YES`.
- Parse failure được xử lý bảo thủ: **KEEP/YES** để tránh xóa nhầm target thật.


In [ ]:
# ============================================================
# CELL 1 — CONFIG
# ============================================================

from pathlib import Path
import random

SEED = 42
random.seed(SEED)

RUN_MODE = "FULL"       # chạy SMOKE trước, ổn rồi đổi "FULL"

# CountGD++
COUNTGD_GITHUB = "https://github.com/niki-amini-naieni/CountGDPlusPlus.git"
COUNTGD_REPO = Path("/kaggle/working/CountGDPlusPlus")
COUNTGD_GDRIVE_ID = "1j6N22TtKu2NVcKpgfrf-sJHGeLDqs9hs"
COUNTGD_THRESH = 0.23

# V4 verifier
QWEN_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
CONTACT_SHEET_SIZE = 12
PANEL_SIZE = 224
CROP_PADDING_FRAC = 0.30
KEEP_ON_PARSE_FAILURE = True

# Positive evaluation: 96 unique benchmark images in FULL.
MAX_POS_BOXES_TO_VERIFY_PER_IMAGE = 80
SMOKE_POS_IMAGES = 5

# Hard-negative evaluation: 111 pairs in FULL.
MAX_HARD_BOXES_PER_IMAGE = 120
SMOKE_HARD_OLD = 4
SMOKE_HARD_NEW = 4

SAVE_EVERY_SHEET = 1
FAIL_FAST = False
RETRY_ERRORS = False

OUTPUT_DIR = Path("/kaggle/working/nckh_v4_qwen3_detection_verification")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HARD_CANDIDATES_CSV = OUTPUT_DIR / "v4_hard_candidates.csv"
POS_SELECTION_CSV = OUTPUT_DIR / "v4_positive_selection.csv"
CROP_MANIFEST_CSV = OUTPUT_DIR / "v4_crop_manifest.csv"
CROP_PREDICTIONS_CSV = OUTPUT_DIR / "v4_qwen3_predictions.csv"
BOX_METRICS_JSON = OUTPUT_DIR / "v4_box_metrics.json"
HARD_IMAGE_METRICS_CSV = OUTPUT_DIR / "v4_hard_image_metrics.csv"
POS_IMAGE_METRICS_CSV = OUTPUT_DIR / "v4_positive_image_metrics.csv"
FAILURE_CASES_CSV = OUTPUT_DIR / "v4_failure_cases.csv"
SUMMARY_JSON = OUTPUT_DIR / "v4_summary.json"

print("RUN_MODE:", RUN_MODE)
print("QWEN_MODEL_ID:", QWEN_MODEL_ID)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# ============================================================
# CELL 2 — DEPENDENCIES + COUNTGD++ REPO
# Giữ Torch/Torchvision hiện tại.
# ============================================================

import sys
import subprocess
import importlib.metadata as im
from pathlib import Path

def pkg_version(name):
    try:
        return im.version(name)
    except Exception:
        return None

print("Existing torch:", pkg_version("torch"))
print("Existing torchvision:", pkg_version("torchvision"))
print("Existing transformers:", pkg_version("transformers"))

if not COUNTGD_REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", COUNTGD_GITHUB, str(COUNTGD_REPO)],
        check=True,
    )

req_file = COUNTGD_REPO / "requirements.txt"
if req_file.exists():
    blocked = (
        "torch", "torchvision", "torchaudio", "numpy", "scipy", "opencv",
        "gradio", "spaces", "deeplake"
    )
    keep = []
    for line in req_file.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if any(s.lower().startswith(x) for x in blocked):
            continue
        keep.append(s)

    filtered = Path("/kaggle/working/countgd_v4_requirements_filtered.txt")
    filtered.write_text("\n".join(keep) + "\n", encoding="utf-8")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered)],
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "transformers>=4.57.0,<5",
        "accelerate>=1.2.0",
        "sentencepiece",
        "gdown",
        "qwen-vl-utils",
    ],
    check=True,
)

print("Dependency setup finished.")
print("transformers:", pkg_version("transformers"))


In [ ]:
# ============================================================
# CELL 3 — VERIFY TORCH/CUDA + PATCH/BUILD GROUNDINGDINO OPS
# ============================================================

import os
import re
import sys
import gc
import importlib
import subprocess
from pathlib import Path

import torch
import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Torch CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("V3 cần GPU Kaggle.")

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

from torchvision.ops import nms
_test_boxes = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]], device="cuda")
_test_scores = torch.tensor([0.9,0.8], device="cuda")
_ = nms(_test_boxes, _test_scores, 0.5)
print("torchvision CUDA NMS: PASS")

def detectron2_ok():
    try:
        import detectron2
        from detectron2 import _C
        return True
    except Exception as e:
        print("Detectron2 check:", repr(e))
        return False

if not detectron2_ok():
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
            "git+https://github.com/facebookresearch/detectron2.git",
        ],
        check=True,
    )
    importlib.invalidate_caches()
    if not detectron2_ok():
        raise RuntimeError("Detectron2 vẫn không load được sau khi cài.")

OPS_DIR = COUNTGD_REPO / "models" / "GroundingDINO" / "ops"
SRC_DIR = OPS_DIR / "src"
assert OPS_DIR.exists(), OPS_DIR

source_suffixes = {".h", ".hpp", ".cpp", ".cu", ".cuh"}
patched = []
for path in SRC_DIR.rglob("*"):
    if not path.is_file() or path.suffix not in source_suffixes:
        continue
    original = path.read_text(encoding="utf-8", errors="ignore")
    updated = original
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.is_cuda\(\)",
        r"\1.is_cuda()",
        updated,
    )
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.scalar_type\(\)",
        r"\1.scalar_type()",
        updated,
    )
    updated = updated.replace("value.type()", "value.scalar_type()")
    if updated != original:
        path.write_text(updated, encoding="utf-8")
        patched.append(str(path.relative_to(OPS_DIR)))

print("Patched files:", patched if patched else "already compatible")

from torch.utils.cpp_extension import CUDA_HOME
if CUDA_HOME:
    os.environ["CUDA_HOME"] = str(CUDA_HOME)

major, minor = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
os.environ["MAX_JOBS"] = "1"

if Path("/usr/bin/gcc-11").exists():
    os.environ["CC"] = "/usr/bin/gcc-11"
if Path("/usr/bin/g++-11").exists():
    os.environ["CXX"] = "/usr/bin/g++-11"

def try_import_msda():
    try:
        if str(OPS_DIR) not in sys.path:
            sys.path.insert(0, str(OPS_DIR))
        importlib.invalidate_caches()
        return importlib.import_module("MultiScaleDeformableAttention")
    except Exception:
        return None

MSDA = try_import_msda()
if MSDA is None:
    print("Building GroundingDINO CUDA ops...")
    subprocess.run(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=str(OPS_DIR),
        check=True,
    )
    importlib.invalidate_caches()
    sys.modules.pop("MultiScaleDeformableAttention", None)
    MSDA = try_import_msda()

if MSDA is None:
    raise RuntimeError("Không import được MultiScaleDeformableAttention sau build.")

print("GroundingDINO CUDA ops: PASS", MSDA.__file__)


In [ ]:
# ============================================================
# CELL 4 — DOWNLOAD COUNTGD++ WEIGHTS
# ============================================================

import sys
import subprocess
from pathlib import Path

CHECKPOINT_DIR = COUNTGD_REPO / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BERT_DIR = CHECKPOINT_DIR / "bert-base-uncased"
if not (BERT_DIR / "config.json").exists():
    print("Downloading bert-base-uncased...")
    subprocess.run(
        [sys.executable, "download_bert.py"],
        cwd=str(COUNTGD_REPO),
        check=True,
    )

COUNTGD_CKPT = CHECKPOINT_DIR / "countgd_plusplus.pth"
if (not COUNTGD_CKPT.exists()) or COUNTGD_CKPT.stat().st_size < 500 * 1024**2:
    print("Downloading CountGD++ checkpoint...")
    subprocess.run(
        [sys.executable, "-m", "gdown", COUNTGD_GDRIVE_ID, "-O", str(COUNTGD_CKPT)],
        check=True,
    )

assert (BERT_DIR / "config.json").exists()
assert COUNTGD_CKPT.exists()
print("BERT:", BERT_DIR)
print("CountGD++ checkpoint GB:", round(COUNTGD_CKPT.stat().st_size / 1024**3, 2))


## Input cần Add vào Kaggle trước Cell 5

V4 cần:

1. FSC-147 như V1/V2/V3, có:
   - `images_384_VarV2`
   - `annotation_FSC147_384.json`

2. ZIP kết quả V1 để lấy candidate boxes của positive query:
   - `nckh_fsc147_full_count_refinement_export.zip`

3. Benchmark mới của nhóm trưởng:
   - **`FSC147_Presence_verified_absent_111.csv`**

V4 không cần ZIP V2/V3.


In [ ]:
# ============================================================
# CELL 5 — FIND INPUTS + LOAD 111 BENCHMARK + V1 POSITIVE BOXES
# ============================================================

from pathlib import Path
import zipfile
import json
import pandas as pd
import numpy as np

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_file(name):
    hits = []
    for root in SEARCH_ROOTS:
        if root.exists():
            hits.extend([p for p in root.rglob(name) if p.is_file()])
    if not hits:
        return None
    return sorted(hits, key=lambda p: (0 if str(p).startswith("/kaggle/input") else 1, len(str(p))))[0]

def find_image_dir():
    hits = []
    for root in SEARCH_ROOTS:
        if root.exists():
            hits.extend([p for p in root.rglob("images_384_VarV2") if p.is_dir()])
    if not hits:
        return None
    return max(hits, key=lambda p: len(list(p.glob("*.jpg"))))

IMAGE_DIR = find_image_dir()
ANNOTATION_FILE = find_file("annotation_FSC147_384.json")
BENCHMARK_FILE = find_file("FSC147_Presence_verified_absent_111.csv")

if IMAGE_DIR is None:
    raise FileNotFoundError("Không tìm thấy images_384_VarV2")
if ANNOTATION_FILE is None:
    raise FileNotFoundError("Không tìm thấy annotation_FSC147_384.json")
if BENCHMARK_FILE is None:
    raise FileNotFoundError("Không tìm thấy FSC147_Presence_verified_absent_111.csv")

image_index = {p.name: p for p in IMAGE_DIR.glob("*.jpg")}
with open(ANNOTATION_FILE, "r", encoding="utf-8") as f:
    annotations = json.load(f)

# ---------- load / validate 111 benchmark ----------
bench = pd.read_csv(BENCHMARK_FILE)
required = {
    "pair_id","image_name","source_class","query_target","presence_label","count_target",
    "countgdpp_pred","t2icount_pred","source_gt_count","origin_batch",
    "assistant_self_check","final_decision","benchmark_inclusion"
}
missing = sorted(required - set(bench.columns))
if missing:
    raise ValueError("Benchmark thiếu cột: " + str(missing))

bench = bench[bench["benchmark_inclusion"].astype(str).str.upper().eq("YES")].copy()
for c in ["presence_label","count_target","countgdpp_pred","t2icount_pred","source_gt_count"]:
    bench[c] = pd.to_numeric(bench[c], errors="coerce")

assert len(bench) == 111, len(bench)
assert (bench["presence_label"] == 0).all()
assert (bench["count_target"] == 0).all()
assert bench["final_decision"].astype(str).eq("KEEP_NEGATIVE").all()

bench["image_path"] = bench["image_name"].map(lambda n: str(image_index.get(str(n), "")))
if bench["image_path"].eq("").any():
    raise FileNotFoundError("Một số ảnh benchmark không có trong IMAGE_DIR")

print("=== V4 BENCHMARK ===")
print("Pairs:", len(bench))
print("Unique images:", bench["image_name"].nunique())
print("Source classes:", bench["source_class"].nunique())
print("Absent targets:", bench["query_target"].nunique())
print(bench["origin_batch"].value_counts())
print("Provided CountGD++ negative MAE:", float(bench["countgdpp_pred"].abs().mean()))
print("Provided T2ICount negative MAE:", float(bench["t2icount_pred"].abs().mean()))

# ---------- load V1 cached positive candidate boxes ----------
V1_CANDIDATES = find_file("countgd_candidates_shard_00_of_01.csv")
if V1_CANDIDATES is None:
    V1_ZIP = find_file("nckh_fsc147_full_count_refinement_export.zip")
    if V1_ZIP is None:
        raise FileNotFoundError("Thiếu ZIP V1")
    exdir = Path("/kaggle/working/v1_export_v4")
    exdir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(V1_ZIP, "r") as zf:
        zf.extractall(exdir)
    found = list(exdir.rglob("countgd_candidates_shard_00_of_01.csv"))
    if not found:
        raise FileNotFoundError("ZIP V1 thiếu countgd_candidates_shard_00_of_01.csv")
    V1_CANDIDATES = found[0]

cand = pd.read_csv(V1_CANDIDATES)
cand = cand[cand["status"].astype(str).eq("ok")].copy()
cand["image_path"] = cand["image_name"].map(lambda n: str(image_index.get(str(n), "")))

# Positive = same 96 images, correct source_class target.
pos_keys = bench[["image_name","source_class","source_gt_count"]].drop_duplicates().copy()
pos_keys = pos_keys.rename(columns={"source_class":"target","source_gt_count":"benchmark_source_gt_count"})

pos_sel = pos_keys.merge(cand, on=["image_name","target"], how="left", suffixes=("","_v1"))
pos_sel = pos_sel[pos_sel["boxes_json"].notna() & pos_sel["scores_json"].notna()].copy()
pos_sel["gt_count"] = pd.to_numeric(pos_sel["benchmark_source_gt_count"], errors="coerce")
pos_sel["image_path"] = pos_sel["image_name"].map(lambda n: str(image_index.get(str(n), "")))

if RUN_MODE == "SMOKE":
    pos_sel = pos_sel.sample(n=min(SMOKE_POS_IMAGES, len(pos_sel)), random_state=SEED).sort_values("image_name").reset_index(drop=True)
else:
    pos_sel = pos_sel.sort_values(["target","image_name"]).reset_index(drop=True)

pos_sel.to_csv(POS_SELECTION_CSV, index=False)

# Hard manifest aliases so later V3 cells remain compatible.
hard_manifest = bench.copy()
hard_manifest["raw_count"] = hard_manifest["countgdpp_pred"].round().astype(int)
hard_manifest["main_visible_confounder"] = hard_manifest["source_class"].astype(str)

if RUN_MODE == "SMOKE":
    old = hard_manifest[hard_manifest["origin_batch"].eq("V3_OLD_56")]
    new = hard_manifest[hard_manifest["origin_batch"].eq("V4_NEW_REVIEWED")]
    hard_run = pd.concat([
        old.sample(n=min(SMOKE_HARD_OLD, len(old)), random_state=SEED),
        new.sample(n=min(SMOKE_HARD_NEW, len(new)), random_state=SEED+1),
    ], ignore_index=True).sort_values(["origin_batch","pair_id"]).reset_index(drop=True)
else:
    hard_run = hard_manifest.sort_values(["origin_batch","pair_id"]).reset_index(drop=True)

print("Positive images this run:", len(pos_sel))
print("Hard pairs this run:", len(hard_run))
print(hard_run["origin_batch"].value_counts())
display(hard_run[["pair_id","image_name","source_class","query_target","origin_batch"]].head())


In [ ]:
# ============================================================
# CELL 6 — LOAD COUNTGD++ (only for hard-negative candidate boxes)
# ============================================================

import os
import sys
import gc
import importlib
import torch
from pathlib import Path

repo_str = str(COUNTGD_REPO)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

os.chdir(repo_str)

for name in list(sys.modules.keys()):
    if name == "models" or name.startswith("models."):
        del sys.modules[name]

importlib.invalidate_caches()
gc.collect()
torch.cuda.empty_cache()

process_file = COUNTGD_REPO / "process_folder.py"
source = process_file.read_text(encoding="utf-8")
marker = "logging.basicConfig(level=logging.INFO"
if marker not in source:
    raise RuntimeError("process_folder.py structure changed")

library_code = source.split(marker, 1)[0]
countgd_namespace = {"__name__": "countgd_runtime"}
exec(library_code, countgd_namespace)

needed = ["build_arg_parser", "build_model_and_transforms", "count"]
missing = [x for x in needed if x not in countgd_namespace]
if missing:
    raise RuntimeError("Missing CountGD functions: " + str(missing))

parser = countgd_namespace["build_arg_parser"]()
args = parser.parse_args([])
args.device = "cuda:0"
args.pretrain_model_path = str(COUNTGD_CKPT)

_orig_torch_load = torch.load
def _trusted_torch_load(*a, **kw):
    kw.setdefault("weights_only", False)
    return _orig_torch_load(*a, **kw)

torch.load = _trusted_torch_load
try:
    countgd_model, countgd_transform = countgd_namespace["build_model_and_transforms"](args)
finally:
    torch.load = _orig_torch_load

countgd_device = torch.device("cuda:0")
countgd_model = countgd_model.to(countgd_device)
countgd_model.eval()

countgd_namespace["model"] = countgd_model
countgd_namespace["transform"] = countgd_transform
countgd_namespace["device"] = countgd_device

print("CountGD++ loaded on", countgd_device)


In [ ]:
# ============================================================
# CELL 7 — RUN COUNTGD++ ON AUDITED HARD NEGATIVES + RESUME
# ============================================================

from PIL import Image
import pandas as pd
import json
import time
from pathlib import Path

def atomic_csv(df, path):
    tmp = Path(str(path) + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def countgd_predict(image_path, target, conf_thresh=COUNTGD_THRESH):
    image = Image.open(image_path).convert("RGB")
    target = str(target).replace(".", "").strip()
    positive_prompts = {"image": image, "points": []}

    boxes, scores, _ = countgd_namespace["count"](
        image,
        target,
        positive_prompts,
        [],
        [],
        float(conf_thresh),
        False,
    )

    boxes = boxes.tolist() if hasattr(boxes, "tolist") else list(boxes)
    scores = scores.tolist() if hasattr(scores, "tolist") else list(scores)

    return {
        "boxes": [[float(v) for v in b] for b in boxes],
        "scores": [float(x) for x in scores],
        "count": int(len(boxes)),
    }

existing = pd.read_csv(HARD_CANDIDATES_CSV) if HARD_CANDIDATES_CSV.exists() else pd.DataFrame()
records = existing.to_dict("records") if len(existing) else []

done = set()
if len(existing):
    for _, r in existing.iterrows():
        if RETRY_ERRORS:
            if str(r.get("status","")) == "ok":
                done.add(str(r["pair_id"]))
        else:
            done.add(str(r["pair_id"]))

print("Existing hard rows:", len(records))

for idx, row in hard_run.iterrows():
    pair_id = str(row["pair_id"])
    if pair_id in done:
        continue

    t0 = time.time()
    try:
        out = countgd_predict(row["image_path"], row["query_target"], COUNTGD_THRESH)
        status = "ok"
        error = ""
    except Exception as e:
        out = {"boxes": [], "scores": [], "count": 0}
        status = "error"
        error = repr(e)
        if FAIL_FAST:
            raise

    records.append({
        "pair_id": pair_id,
        "image_name": row["image_name"],
        "image_path": row["image_path"],
        "source_class": row["source_class"],
        "target": row["query_target"],
        "oracle_confounder": row["main_visible_confounder"],
        "origin_batch": row["origin_batch"],
        "audit_raw_count": int(row["raw_count"]),
        "countgd_count_023": int(out["count"]),
        "boxes_json": json.dumps(out["boxes"]),
        "scores_json": json.dumps(out["scores"]),
        "latency_sec": round(time.time() - t0, 3),
        "status": status,
        "error": error,
    })
    done.add(pair_id)

    if len(records) % 1 == 0:
        atomic_csv(pd.DataFrame(records), HARD_CANDIDATES_CSV)

    print(
        f"HARD [{idx+1}/{len(hard_run)}] {pair_id} | "
        f"audit raw={row['raw_count']} | rerun={out['count']}"
    )

hard_candidates = pd.DataFrame(records)
atomic_csv(hard_candidates, HARD_CANDIDATES_CSV)

ok_hard = hard_candidates[hard_candidates["status"].astype(str) == "ok"].copy()
if len(ok_hard):
    repro_mae = (
        pd.to_numeric(ok_hard["countgd_count_023"])
        - pd.to_numeric(ok_hard["audit_raw_count"])
    ).abs().mean()
    print("Hard-negative CountGD reproduction MAE:", repro_mae)

print("Saved:", HARD_CANDIDATES_CSV)


In [ ]:
# ============================================================
# CELL 8 — UNLOAD COUNTGD++ BEFORE QWEN
# ============================================================

import gc
import torch

try:
    countgd_namespace["model"] = None
except Exception:
    pass

for name in ["countgd_model", "countgd_transform"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("CountGD unloaded.")
print(
    "CUDA allocated GB:",
    round(torch.cuda.memory_allocated() / 1024**3, 3)
)


In [ ]:
# ============================================================
# CELL 9 — BUILD CROP MANIFEST
# Positive: cached V1 boxes at score > 0.23
# Hard negative: rerun CountGD boxes at 0.23
# ============================================================

import json
import math
import pandas as pd
import numpy as np
from pathlib import Path

def parse_json_list(x):
    try:
        y = json.loads(x) if isinstance(x, str) else x
        return y if isinstance(y, list) else []
    except Exception:
        return []

def clip_box(box, width, height):
    x1, y1, x2, y2 = [float(v) for v in box]
    x1 = max(0.0, min(float(width), x1))
    y1 = max(0.0, min(float(height), y1))
    x2 = max(0.0, min(float(width), x2))
    y2 = max(0.0, min(float(height), y2))
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    return [x1,y1,x2,y2]

def greedy_match_boxes_to_points(boxes, scores, points):
    # one GT point can match at most one predicted box
    used = set()
    matches = [None] * len(boxes)
    order = sorted(range(len(boxes)), key=lambda i: float(scores[i]), reverse=True)

    pts = []
    for p in points:
        if isinstance(p, (list, tuple)) and len(p) >= 2:
            pts.append((float(p[0]), float(p[1])))

    for i in order:
        x1,y1,x2,y2 = boxes[i]
        cx, cy = (x1+x2)/2.0, (y1+y2)/2.0

        candidates = []
        for j, (px,py) in enumerate(pts):
            if j in used:
                continue
            if x1 <= px <= x2 and y1 <= py <= y2:
                d2 = (px-cx)**2 + (py-cy)**2
                candidates.append((d2,j))

        if candidates:
            _, j = min(candidates)
            used.add(j)
            matches[i] = j

    return matches

crop_rows = []

# ---------- POSITIVE ----------
for _, row in pos_sel.iterrows():
    image_name = str(row["image_name"])
    img_path = str(row["image_path"])

    from PIL import Image
    with Image.open(img_path) as im:
        W,H = im.size

    boxes_all = parse_json_list(row["boxes_json"])
    scores_all = parse_json_list(row["scores_json"])

    keep_idx = [
        i for i,s in enumerate(scores_all)
        if float(s) > float(COUNTGD_THRESH) and i < len(boxes_all)
    ]
    # sort high confidence first
    keep_idx = sorted(keep_idx, key=lambda i: float(scores_all[i]), reverse=True)

    boxes = [clip_box(boxes_all[i], W, H) for i in keep_idx]
    scores = [float(scores_all[i]) for i in keep_idx]

    pts = annotations.get(image_name, {}).get("points", [])
    matches = greedy_match_boxes_to_points(boxes, scores, pts)

    # verify at most MAX_POS_BOXES...; the rest are conservatively kept later
    for local_i, (box, score, m) in enumerate(zip(boxes, scores, matches)):
        if local_i >= MAX_POS_BOXES_TO_VERIFY_PER_IMAGE:
            break
        crop_rows.append({
            "crop_id": f"POS__{image_name}__{local_i:04d}",
            "group_id": f"POS__{image_name}",
            "pair_type": "POSITIVE",
            "origin_batch": "POSITIVE",
            "image_name": image_name,
            "image_path": img_path,
            "target": str(row["target"]),
            "source_class": str(row["target"]),
            "box_index": int(local_i),
            "box_json": json.dumps(box),
            "score": float(score),
            "approx_gt_match": int(m is not None),
            "expected_label": "YES" if m is not None else "UNKNOWN",
            "baseline_count": int(len(boxes)),
            "gt_count": int(row["gt_count"]),
        })

# ---------- HARD NEGATIVE ----------
ok_hard = hard_candidates[hard_candidates["status"].astype(str) == "ok"].copy()

for _, row in ok_hard.iterrows():
    boxes = parse_json_list(row["boxes_json"])
    scores = parse_json_list(row["scores_json"])
    order = sorted(
        range(min(len(boxes), len(scores))),
        key=lambda i: float(scores[i]),
        reverse=True,
    )
    order = order[:MAX_HARD_BOXES_PER_IMAGE]

    from PIL import Image
    with Image.open(row["image_path"]) as im:
        W,H = im.size

    for rank_i, i in enumerate(order):
        box = clip_box(boxes[i], W, H)
        crop_rows.append({
            "crop_id": f"{row['pair_id']}__{rank_i:04d}",
            "group_id": str(row["pair_id"]),
            "pair_type": "HARD_NEGATIVE",
            "origin_batch": str(row.get("origin_batch", "UNKNOWN")),
            "image_name": str(row["image_name"]),
            "image_path": str(row["image_path"]),
            "target": str(row["target"]),
            "source_class": str(row["source_class"]),
            "box_index": int(i),
            "box_json": json.dumps(box),
            "score": float(scores[i]),
            "approx_gt_match": 0,
            "expected_label": "NO",
            "baseline_count": int(row["countgd_count_023"]),
            "gt_count": 0,
        })

crop_manifest = pd.DataFrame(crop_rows)
crop_manifest.to_csv(CROP_MANIFEST_CSV, index=False)

print("Crop rows:", len(crop_manifest))
print(crop_manifest["pair_type"].value_counts())
print("Approx positive GT-matched boxes:", int(crop_manifest["approx_gt_match"].sum()))
print("Saved:", CROP_MANIFEST_CSV)
display(crop_manifest.head())


In [ ]:
# ============================================================
# CELL 10 — LOAD QWEN3-VL-4B
# ============================================================

import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

QWEN_DEVICE = "cuda:0"

qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
qwen_model.eval()

print("Loaded:", QWEN_MODEL_ID)
print("Model device:", next(qwen_model.parameters()).device)


In [ ]:
# ============================================================
# CELL 11 — CROP / CONTACT-SHEET / QWEN VERIFIER UTILITIES
# ============================================================

from PIL import Image, ImageDraw, ImageFont, ImageOps
import pandas as pd
import numpy as np
import json
import re
import math
import torch

def expanded_crop(image, box, padding_frac=CROP_PADDING_FRAC):
    W,H = image.size
    x1,y1,x2,y2 = [float(v) for v in box]
    bw = max(1.0, x2-x1)
    bh = max(1.0, y2-y1)
    px = bw * padding_frac
    py = bh * padding_frac
    xx1 = int(max(0, math.floor(x1-px)))
    yy1 = int(max(0, math.floor(y1-py)))
    xx2 = int(min(W, math.ceil(x2+px)))
    yy2 = int(min(H, math.ceil(y2+py)))
    crop = image.crop((xx1,yy1,xx2,yy2)).convert("RGB")
    return crop

def make_contact_sheet(image, rows):
    n = len(rows)
    cols = 4
    rows_n = int(math.ceil(n / cols))
    label_h = 28
    sheet = Image.new(
        "RGB",
        (cols * PANEL_SIZE, rows_n * (PANEL_SIZE + label_h)),
        "white",
    )
    draw = ImageDraw.Draw(sheet)

    for local_id, row in enumerate(rows):
        box = json.loads(row["box_json"])
        crop = expanded_crop(image, box)
        crop = ImageOps.contain(crop, (PANEL_SIZE, PANEL_SIZE))

        x0 = (local_id % cols) * PANEL_SIZE
        y0 = (local_id // cols) * (PANEL_SIZE + label_h)

        panel = Image.new("RGB", (PANEL_SIZE, PANEL_SIZE), "white")
        px = (PANEL_SIZE - crop.width) // 2
        py = (PANEL_SIZE - crop.height) // 2
        panel.paste(crop, (px,py))
        sheet.paste(panel, (x0,y0))

        draw.rectangle(
            [x0, y0, x0+PANEL_SIZE-1, y0+PANEL_SIZE+label_h-1],
            outline="black",
            width=2,
        )
        draw.rectangle(
            [x0, y0+PANEL_SIZE, x0+PANEL_SIZE-1, y0+PANEL_SIZE+label_h-1],
            fill="white",
        )
        draw.text(
            (x0+6, y0+PANEL_SIZE+5),
            f"ID {local_id}",
            fill="black",
        )

    return sheet

LINE_RE = re.compile(r"^\s*(?:ID\s*)?(\d+)\s*\|\s*(YES|NO)\s*$", re.I)

def parse_sheet_output(text, n):
    parsed = {}
    for line in str(text).splitlines():
        m = LINE_RE.match(line.strip())
        if m:
            idx = int(m.group(1))
            if 0 <= idx < n:
                parsed[idx] = m.group(2).upper()
    return parsed

@torch.inference_mode()
def qwen_verify_sheet(image, target, sheet_rows):
    sheet = make_contact_sheet(image, sheet_rows)

    prompt = f"""
The second image is a numbered contact sheet of candidate detections from an object-counting model.

Target category: {target}

Task:
For EACH numbered panel, decide whether the main object shown in that panel is truly an instance of the target category "{target}".

Important:
- Judge the object in the numbered crop, not merely whether the target exists somewhere in the whole scene.
- Similar-looking or semantically related objects are NOT the target.
- Be conservative about rejecting a real target.
- Output exactly one line per panel and nothing else.
- Format exactly: 0|YES or 0|NO (use the numeric panel id; do not write the word ID)

Example:
0|YES
1|NO
2|YES
""".strip()

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "image", "image": sheet},
            {"type": "text", "text": prompt},
        ],
    }]

    text_prompt = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        image_patch_size=16,
        return_video_kwargs=True,
        return_video_metadata=True,
    )

    video_metadatas = None
    if video_inputs is not None:
        video_inputs, video_metadatas = zip(*video_inputs)
        video_inputs = list(video_inputs)
        video_metadatas = list(video_metadatas)

    inputs = qwen_processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        video_metadata=video_metadatas,
        padding=True,
        return_tensors="pt",
        do_resize=False,
        **video_kwargs,
    )
    inputs = inputs.to(next(qwen_model.parameters()).device)

    generated_ids = qwen_model.generate(
        **inputs,
        max_new_tokens=max(64, 10 * len(sheet_rows) + 32),
        do_sample=False,
    )

    trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = qwen_processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return output_text, sheet

print("Verifier utilities ready.")


In [ ]:
# ============================================================
# CELL 12 — SANITY CHECK ONE CONTACT SHEET
# ============================================================

if len(crop_manifest) == 0:
    raise RuntimeError("Crop manifest rỗng.")

g0 = crop_manifest.groupby(["group_id","target"], sort=False).first().reset_index().iloc[0]
group = crop_manifest[
    (crop_manifest["group_id"] == g0["group_id"])
    & (crop_manifest["target"] == g0["target"])
].head(CONTACT_SHEET_SIZE)

image = Image.open(group.iloc[0]["image_path"]).convert("RGB")
raw, sheet = qwen_verify_sheet(image, str(g0["target"]), group.to_dict("records"))

print("Group:", g0["group_id"])
print("Target:", g0["target"])
print("Raw Qwen output:")
print(raw)
print("Parsed:", parse_sheet_output(raw, len(group)))
display(sheet)

print("SANITY CHECK COMPLETE")


In [ ]:
# ============================================================
# CELL 13 — RUN DETECTION-LEVEL QWEN VERIFICATION + RESUME
# ============================================================

import pandas as pd
import json
import time
from pathlib import Path

existing = pd.read_csv(CROP_PREDICTIONS_CSV) if CROP_PREDICTIONS_CSV.exists() else pd.DataFrame()
records = existing.to_dict("records") if len(existing) else []

done = set()
if len(existing):
    if RETRY_ERRORS:
        done = set(
            existing.loc[existing["status"].astype(str) == "ok", "crop_id"].astype(str)
        )
    else:
        done = set(existing["crop_id"].astype(str))

sheet_counter = 0

for (group_id, target), g in crop_manifest.groupby(["group_id","target"], sort=False):
    g = g.copy()
    g = g[~g["crop_id"].astype(str).isin(done)].copy()

    if len(g) == 0:
        continue

    image = Image.open(g.iloc[0]["image_path"]).convert("RGB")

    for start in range(0, len(g), CONTACT_SHEET_SIZE):
        chunk = g.iloc[start:start+CONTACT_SHEET_SIZE].copy()
        chunk_rows = chunk.to_dict("records")
        t0 = time.time()

        try:
            raw_output, _ = qwen_verify_sheet(image, str(target), chunk_rows)
            parsed = parse_sheet_output(raw_output, len(chunk_rows))
            status = "ok"
            error = ""
        except Exception as e:
            raw_output = ""
            parsed = {}
            status = "error"
            error = repr(e)
            if FAIL_FAST:
                raise

        for local_id, row in enumerate(chunk_rows):
            pred = parsed.get(local_id)

            if pred is None:
                if KEEP_ON_PARSE_FAILURE:
                    pred = "YES"
                    parse_fallback = 1
                else:
                    pred = "INVALID"
                    parse_fallback = 0
            else:
                parse_fallback = 0

            records.append({
                **row,
                "vlm_pred": pred,
                "vlm_keep": int(pred == "YES"),
                "parse_fallback": int(parse_fallback),
                "raw_output": raw_output,
                "sheet_local_id": int(local_id),
                "latency_sheet_sec": round(time.time() - t0, 3),
                "status": status,
                "error": error,
            })
            done.add(str(row["crop_id"]))

        sheet_counter += 1
        if sheet_counter % SAVE_EVERY_SHEET == 0:
            atomic_csv(pd.DataFrame(records), CROP_PREDICTIONS_CSV)

        print(
            f"SHEET {sheet_counter} | {group_id} | target={target} | "
            f"boxes={len(chunk_rows)} | parsed={len(parsed)}/{len(chunk_rows)}"
        )

pred_df = pd.DataFrame(records)
atomic_csv(pred_df, CROP_PREDICTIONS_CSV)

print("Predictions:", len(pred_df))
print("Fallback parses:", int(pred_df["parse_fallback"].sum()) if len(pred_df) else 0)
print("Saved:", CROP_PREDICTIONS_CSV)


In [ ]:
# ============================================================
# CELL 14 — BOX-LEVEL METRICS
# ============================================================

import json
import numpy as np
import pandas as pd

pred = pd.read_csv(CROP_PREDICTIONS_CSV)
pred = pred[pred["status"].astype(str) == "ok"].copy()
pred["vlm_keep"] = pd.to_numeric(pred["vlm_keep"], errors="coerce").fillna(1).astype(int)
pred["approx_gt_match"] = pd.to_numeric(pred["approx_gt_match"], errors="coerce").fillna(0).astype(int)

# Positive main metric only uses boxes approximately matched to a GT point.
pos_true = pred[
    (pred["pair_type"] == "POSITIVE")
    & (pred["approx_gt_match"] == 1)
].copy()

hard_false = pred[
    pred["pair_type"] == "HARD_NEGATIVE"
].copy()

positive_box_retention = float(pos_true["vlm_keep"].mean()) if len(pos_true) else np.nan
hard_box_rejection = float((hard_false["vlm_keep"] == 0).mean()) if len(hard_false) else np.nan
hard_box_false_keep = float(hard_false["vlm_keep"].mean()) if len(hard_false) else np.nan

# Descriptive only: unmatched CountGD boxes on positive images are not guaranteed false
pos_unmatched = pred[
    (pred["pair_type"] == "POSITIVE")
    & (pred["approx_gt_match"] == 0)
].copy()
positive_unmatched_rejection = (
    float((pos_unmatched["vlm_keep"] == 0).mean())
    if len(pos_unmatched) else np.nan
)

box_metrics = {
    "positive_gt_matched_boxes_n": int(len(pos_true)),
    "positive_box_retention": positive_box_retention,
    "positive_box_false_reject_rate": (
        float(1.0 - positive_box_retention)
        if np.isfinite(positive_box_retention) else None
    ),
    "hard_negative_boxes_n": int(len(hard_false)),
    "hard_negative_box_rejection": hard_box_rejection,
    "hard_negative_box_false_keep_rate": hard_box_false_keep,
    "hard_negative_box_rejection_old56": (
        float((hard_false.loc[hard_false["origin_batch"].astype(str)=="V3_OLD_56", "vlm_keep"] == 0).mean())
        if len(hard_false.loc[hard_false["origin_batch"].astype(str)=="V3_OLD_56"]) else None
    ),
    "hard_negative_box_rejection_new55": (
        float((hard_false.loc[hard_false["origin_batch"].astype(str)=="V4_NEW_REVIEWED", "vlm_keep"] == 0).mean())
        if len(hard_false.loc[hard_false["origin_batch"].astype(str)=="V4_NEW_REVIEWED"]) else None
    ),
    "positive_unmatched_boxes_n": int(len(pos_unmatched)),
    "positive_unmatched_rejection_descriptive_only": (
        positive_unmatched_rejection
        if np.isfinite(positive_unmatched_rejection) else None
    ),
    "parse_fallback_count": int(pred["parse_fallback"].sum()),
}

with open(BOX_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(box_metrics, f, indent=2, ensure_ascii=False)

print(json.dumps(box_metrics, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# CELL 15 — IMAGE-LEVEL SIMULATION
# Hard negatives: verify all rerun boxes (up to configured cap)
# Positives: verified boxes filtered; unverified boxes conservatively kept
# ============================================================

import pandas as pd
import numpy as np
import json

pred = pd.read_csv(CROP_PREDICTIONS_CSV)
pred = pred[pred["status"].astype(str) == "ok"].copy()
pred["vlm_keep"] = pd.to_numeric(pred["vlm_keep"], errors="coerce").fillna(1).astype(int)

# ----- HARD NEGATIVE -----
hard_rows = []
for _, base in hard_candidates[hard_candidates["status"].astype(str)=="ok"].iterrows():
    gid = str(base["pair_id"])
    g = pred[pred["group_id"].astype(str) == gid]

    baseline_count = int(base["countgd_count_023"])
    verified_n = int(len(g))
    kept_verified = int(g["vlm_keep"].sum())

    # If count > verification cap, unverified detections are conservatively kept.
    unverified_n = max(0, baseline_count - verified_n)
    filtered_count = kept_verified + unverified_n

    hard_rows.append({
        "pair_id": gid,
        "image_name": base["image_name"],
        "source_class": base["source_class"],
        "target": base["target"],
        "origin_batch": base.get("origin_batch", "UNKNOWN"),
        "baseline_count": baseline_count,
        "verified_boxes": verified_n,
        "kept_verified": kept_verified,
        "unverified_kept": unverified_n,
        "filtered_count": filtered_count,
        "raw_abs_error": baseline_count,
        "filtered_abs_error": filtered_count,
        "raw_false_activation": int(baseline_count > 0),
        "filtered_false_activation": int(filtered_count > 0),
        "box_rejection_rate": (
            1.0 - kept_verified/verified_n if verified_n > 0 else np.nan
        ),
    })

hard_img = pd.DataFrame(hard_rows)
hard_img.to_csv(HARD_IMAGE_METRICS_CSV, index=False)

# ----- POSITIVE -----
pos_rows = []
for _, base in pos_sel.iterrows():
    gid = f"POS__{base['image_name']}"
    g = pred[pred["group_id"].astype(str) == gid]

    baseline_count = int(base["baseline_count_023"])
    verified_n = int(len(g))
    kept_verified = int(g["vlm_keep"].sum())
    unverified_n = max(0, baseline_count - verified_n)
    filtered_count = kept_verified + unverified_n
    gt = int(base["gt_count"])

    matched = g[pd.to_numeric(g["approx_gt_match"], errors="coerce").fillna(0).astype(int)==1]
    matched_retention = (
        float(matched["vlm_keep"].mean()) if len(matched) else np.nan
    )

    pos_rows.append({
        "image_name": base["image_name"],
        "target": base["target"],
        "gt_count": gt,
        "baseline_count": baseline_count,
        "verified_boxes": verified_n,
        "kept_verified": kept_verified,
        "unverified_kept": unverified_n,
        "filtered_count": filtered_count,
        "baseline_abs_error": abs(baseline_count - gt),
        "filtered_abs_error": abs(filtered_count - gt),
        "positive_survived": int(filtered_count > 0),
        "matched_box_retention": matched_retention,
        "verified_fraction": verified_n / baseline_count if baseline_count > 0 else 0.0,
    })

pos_img = pd.DataFrame(pos_rows)
pos_img.to_csv(POS_IMAGE_METRICS_CSV, index=False)

print("=== HARD NEGATIVE IMAGE LEVEL ===")
if len(hard_img):
    print("N:", len(hard_img))
    print("Raw FPR:", hard_img["raw_false_activation"].mean())
    print("Filtered FPR:", hard_img["filtered_false_activation"].mean())
    print("Raw hard MAE:", hard_img["raw_abs_error"].mean())
    print("Filtered hard MAE:", hard_img["filtered_abs_error"].mean())

print("\n=== POSITIVE IMAGE SUBSET ===")
if len(pos_img):
    print("N:", len(pos_img), "classes:", pos_img["target"].nunique())
    print("Baseline MAE:", pos_img["baseline_abs_error"].mean())
    print("Filtered MAE:", pos_img["filtered_abs_error"].mean())
    print("Positive survival:", pos_img["positive_survived"].mean())
    print("Mean verified fraction:", pos_img["verified_fraction"].mean())

display(hard_img.head())
display(pos_img.head())


In [ ]:
# ============================================================
# CELL 16 — FAILURE CASES + SUMMARY
# ============================================================

import pandas as pd
import numpy as np
import json

pred = pd.read_csv(CROP_PREDICTIONS_CSV)
pred = pred[pred["status"].astype(str) == "ok"].copy()
pred["vlm_keep"] = pd.to_numeric(pred["vlm_keep"], errors="coerce").fillna(1).astype(int)
pred["approx_gt_match"] = pd.to_numeric(pred["approx_gt_match"], errors="coerce").fillna(0).astype(int)

positive_false_rejects = pred[
    (pred["pair_type"] == "POSITIVE")
    & (pred["approx_gt_match"] == 1)
    & (pred["vlm_keep"] == 0)
].copy()
positive_false_rejects["failure_type"] = "POSITIVE_BOX_FALSE_REJECT"

hard_false_keeps = pred[
    (pred["pair_type"] == "HARD_NEGATIVE")
    & (pred["vlm_keep"] == 1)
].copy()
hard_false_keeps["failure_type"] = "HARD_NEGATIVE_BOX_FALSE_KEEP"

failures = pd.concat(
    [positive_false_rejects, hard_false_keeps],
    ignore_index=True,
)
failures.to_csv(FAILURE_CASES_CSV, index=False)

with open(BOX_METRICS_JSON, "r", encoding="utf-8") as f:
    box_metrics = json.load(f)

hard_raw_fpr = float(hard_img["raw_false_activation"].mean()) if len(hard_img) else np.nan
hard_filtered_fpr = float(hard_img["filtered_false_activation"].mean()) if len(hard_img) else np.nan
hard_raw_mae = float(hard_img["raw_abs_error"].mean()) if len(hard_img) else np.nan
hard_filtered_mae = float(hard_img["filtered_abs_error"].mean()) if len(hard_img) else np.nan

pos_baseline_mae = float(pos_img["baseline_abs_error"].mean()) if len(pos_img) else np.nan
pos_filtered_mae = float(pos_img["filtered_abs_error"].mean()) if len(pos_img) else np.nan
pos_survival = float(pos_img["positive_survived"].mean()) if len(pos_img) else np.nan


# ---------- V4 OLD_56 vs NEW_55 hard-negative split ----------
origin_metrics = {}
if "origin_batch" in hard_img.columns:
    for origin, tag in [("V3_OLD_56","OLD_56"),("V4_NEW_REVIEWED","NEW_55")]:
        g = hard_img[hard_img["origin_batch"].astype(str) == origin].copy()
        if len(g):
            origin_metrics[tag] = {
                "n": int(len(g)),
                "raw_fpr": float((g["baseline_count"] > 0).mean()),
                "filtered_fpr": float((g["filtered_count"] > 0).mean()),
                "raw_mae": float(g["baseline_count"].abs().mean()),
                "filtered_mae": float(g["filtered_count"].abs().mean()),
            }

summary = {
    "phase": "V4_111_HARD_NEGATIVES_QWEN3_VL",
    "qwen_model": QWEN_MODEL_ID,
    "hard_origin_metrics": origin_metrics,
    "run_mode": RUN_MODE,
    "countgd_threshold": COUNTGD_THRESH,
    "positive_images_n": int(len(pos_img)),
    "positive_classes_n": int(pos_img["target"].nunique()) if len(pos_img) else 0,
    "hard_negative_pairs_n": int(len(hard_img)),
    "box_metrics": box_metrics,
    "hard_image_metrics": {
        "raw_fpr": hard_raw_fpr if np.isfinite(hard_raw_fpr) else None,
        "filtered_fpr": hard_filtered_fpr if np.isfinite(hard_filtered_fpr) else None,
        "raw_mae": hard_raw_mae if np.isfinite(hard_raw_mae) else None,
        "filtered_mae": hard_filtered_mae if np.isfinite(hard_filtered_mae) else None,
    },
    "positive_subset_metrics": {
        "baseline_mae": pos_baseline_mae if np.isfinite(pos_baseline_mae) else None,
        "filtered_mae": pos_filtered_mae if np.isfinite(pos_filtered_mae) else None,
        "positive_survival_rate": pos_survival if np.isfinite(pos_survival) else None,
        "note": (
            "Positive subset uses the benchmark's unique images with the true source_class query. "
            "GT-point matching is approximate. Unverified boxes are conservatively kept."
        ),
    },
    "failure_counts": {
        "positive_box_false_rejects": int(len(positive_false_rejects)),
        "hard_negative_box_false_keeps": int(len(hard_false_keeps)),
    },
    "scientific_note": (
        "V4 uses Qwen3-VL-4B and expands the audited hard-negative benchmark to 111 pairs. "
        "The OLD_56 subset was already observed in V3, so NEW_55 should be reported separately."
    ),
}

with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nFailure cases:", len(failures))
print("Saved:", FAILURE_CASES_CSV)


In [ ]:
# ============================================================
# CELL 17 — OPTIONAL FAILURE CROP GALLERY
# Saves top examples for manual inspection.
# ============================================================

from PIL import Image, ImageDraw, ImageOps
from pathlib import Path
import json
import pandas as pd

GALLERY_DIR = OUTPUT_DIR / "failure_gallery"
GALLERY_DIR.mkdir(parents=True, exist_ok=True)

show_fail = failures.sort_values(
    ["failure_type","score"],
    ascending=[True, False],
).groupby("failure_type", group_keys=False).head(12)

for _, row in show_fail.iterrows():
    image = Image.open(row["image_path"]).convert("RGB")
    box = json.loads(row["box_json"])
    crop = expanded_crop(image, box, padding_frac=0.45)

    canvas = Image.new("RGB", (max(320,crop.width), crop.height+70), "white")
    canvas.paste(crop, ((canvas.width-crop.width)//2, 0))
    draw = ImageDraw.Draw(canvas)
    label = (
        f"{row['failure_type']} | target={row['target']} | "
        f"pred={row['vlm_pred']} | score={float(row['score']):.3f}"
    )
    draw.text((8,crop.height+8), label, fill="black")

    safe_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(row["crop_id"]))
    canvas.save(GALLERY_DIR / f"{safe_id}.jpg", quality=92)

print("Gallery:", GALLERY_DIR)
print("Saved images:", len(list(GALLERY_DIR.glob("*.jpg"))))


In [ ]:
# ============================================================
# CELL 18 — EXPORT RESULT ZIP
# ============================================================

import shutil
from pathlib import Path
from IPython.display import FileLink, display

archive = shutil.make_archive(
    "/kaggle/working/nckh_v4_qwen3_detection_verification_export",
    "zip",
    root_dir=str(OUTPUT_DIR),
)

print("Created:", archive)
print("Size MB:", round(Path(archive).stat().st_size / 1024**2, 2))
display(FileLink(archive))


## Kết quả cần gửi lại để phân tích

Sau khi FULL run, tải:

`nckh_v4_qwen3_detection_verification_export.zip`

Các file quan trọng:

- `v4_summary.json`
- `v4_box_metrics.json`
- `v4_hard_image_metrics.csv`
- `v4_positive_image_metrics.csv`
- `v4_failure_cases.csv`
- `v4_qwen3_predictions.csv`
- `v4_hard_candidates.csv`

### Ưu tiên khi đọc kết quả

1. `NEW_55` hard-negative FPR/MAE — quan trọng vì đây là phần mở rộng mới.
2. Positive box retention.
3. Positive MAE và positive survival.
4. Parse fallback count/rate.
5. Sau đó mới xem `ALL_111` và so với V3 cũ.

Nếu Qwen3 loại hard negatives tốt nhưng vẫn reject nhiều positive boxes, bước tiếp theo mới chuyển sang **Selective Verification / soft rejection**, không hard-filter mọi box.
